In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np
from tqdm import tqdm

from sklearn.model_selection import KFold
from scipy.stats import pearsonr, spearmanr

from analysis_utils import target_testsuites, calculate_pearson_correlation, load_loh_variance_df, load_gmm_df, load_token_entropy_df, convert_to_rank_score
from regression_utils import *

In [2]:
def build_task_df(task, model, n_trials=10, data='ref'):
    variance_df = load_loh_variance_df(model, task)
    tok_ent_df = load_token_entropy_df(model, task)
    lih_gmm_df = load_gmm_df(model, task, data)

    # LOH variance
    var_df = variance_df[['input_index', 'variance']].copy()
    # Token Entropy
    tok_ent_df = tok_ent_df[['input_index', 'average_entropy']].copy()
    # LIH GMM
    lih_df = lih_gmm_df[['input_index', 'logprob', 'test_score']].copy()

    merged = var_df.merge(lih_df, on='input_index', how='inner').merge(tok_ent_df, on='input_index', how='inner')
    merged['successes']  = (merged['test_score'] * n_trials).round().astype(int)
    merged['trials']     = n_trials
    merged['task']       = task
    
    return merged

In [3]:
model = 'llama'
n_trials = 10
tasks = list(target_testsuites.keys())

all_dfs = []
for t in tqdm(tasks, desc='Building tasks'):
    all_dfs.append(build_task_df(t, model=model, data='all'))
all_data = pd.concat(all_dfs, ignore_index=True)
display(all_data.head())

Building tasks:   0%|          | 0/8 [00:00<?, ?it/s]

Building tasks: 100%|██████████| 8/8 [00:45<00:00,  5.65s/it]


,input_index,variance,logprob,test_score,average_entropy,successes,trials,task
0,18919,1.908710,-6.418401,1.0,0.408203,10,10,syntactic_bug_detection
1,12957,29.670761,-50.763356,0.0,1.117188,0,10,syntactic_bug_detection
2,10611,38.251438,1.826626,0.0,0.683594,0,10,syntactic_bug_detection
3,2374,52.772263,-21.094984,0.0,1.250000,0,10,syntactic_bug_detection
4,14912,27.619446,-164.412782,0.0,0.992188,0,10,syntactic_bug_detection


In [4]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

fold_rows = []
coef_rows = []
perm_rows = []

threshold = 0.5

evaluation_approaches = {
    'lih_var_entropy': ['variance','logprob', 'average_entropy'],
    'single_features': ['logprob', 'variance', 'average_entropy'],
}

scaler_type = 'standard' # or minmax

for approach_name, feature_list in evaluation_approaches.items():
    for task in tasks:
        task_df = all_data[all_data['task'] == task].reset_index(drop=True)
        
        for fold_idx, (train_idx, test_idx) in enumerate(kf.split(task_df)):
            train_df = task_df.iloc[train_idx].reset_index(drop=True)
            test_df  = task_df.iloc[test_idx].reset_index(drop=True)
        
            y_train = (train_df['test_score'].values >= threshold).astype(int)
            y_test  = (test_df['test_score'].values  >= threshold).astype(int)
            y_score_test = test_df['test_score'].values

            if approach_name == 'single_features':
                for single_feature in feature_list:
                    if single_feature == 'logprob':
                        spearman_r, spearman_pval = spearmanr(test_df['logprob'], test_df['test_score'])
                    elif single_feature == 'variance':
                        spearman_r, spearman_pval = spearmanr(test_df['variance'], test_df['test_score'])
                        spearman_r = -spearman_r
                    elif single_feature == 'average_entropy':
                        spearman_r, spearman_pval = spearmanr(test_df['average_entropy'], test_df['test_score'])
                        spearman_r = -spearman_r
                    else:
                        raise ValueError(f"Unknown feature: {single_feature}")
                    
                    fold_rows.append({
                        'approach': approach_name,
                        'task': task,
                        'model': f'(single) {single_feature}',
                        'spearmanr': spearman_r,
                    })
            else:
                X_train = train_df[feature_list]
                X_test  = test_df[feature_list]

                model = fit_logistic_model(X_train, y_train, scaler_type=scaler_type)
                p_test = predict_prob(model, X_test)

                fold_rows.append({
                    'approach': approach_name,
                    'task': task,
                    'model': f'{feature_list}',
                    'spearmanr': spearmanr(p_test, y_score_test)[0]
                })

results_df = pd.DataFrame(fold_rows).sort_values(['task','spearmanr'], ascending=[True, False]).reset_index(drop=True)

In [5]:
summary = (results_df
           .groupby('model', as_index=False)[['spearmanr']]
           .agg(['mean', 'std'])
           .round(3))

summary.columns = ['_'.join(col).strip() if col[1] else col[0] for col in summary.columns]
summary = summary.sort_values('spearmanr_mean')

print("10-Fold Cross Validation Results (aggregated across all tasks and folds)")
print(summary.to_string(index=False))

10-Fold Cross Validation Results (aggregated across all tasks and folds)
                                     model  spearmanr_mean  spearmanr_std
                  (single) average_entropy           0.327          0.127
                          (single) logprob           0.382          0.111
                         (single) variance           0.603          0.190
['variance', 'logprob', 'average_entropy']           0.611          0.144
